# Query Routing Model Training (BM25 T1 vs Full)

This notebook trains a **query-time** router using the JSONL produced by `scripts/generate_query_routing_data.py`.

**Input format (JSONL):** one example per line
```json
{"qid": "123", "y": 0, "x": [ ... ]}
```

- `y=0` → stay in Tier-1 (fast path)
- `y=1` → fall through (use full index / Tier-2)

The notebook:
- trains a lightweight supervised model (logistic regression)
- evaluates it on validation and test splits
- saves the trained model and feature metadata for reuse at query time

In [ ]:
# Imports, paths, and configs

import os, json
import numpy as np

DATA_DIR = "/kaggle/input/YOUR_DATASET_FOLDER"
TRAIN_JSONL = os.path.join(DATA_DIR, "train_query_routing.jsonl")
FEATURES_JSON = os.path.join(DATA_DIR, "train_query_routing_features.json")

RANDOM_SEED = 42
TEST_FRAC = 0.10
VAL_FRAC  = 0.10  # of total, not of remaining

In [ ]:
# Load data

# Load feature names
with open(FEATURES_JSON, "r") as f:
    feature_names = json.load(f)["feature_names"]

# Load JSONL
rows = []
with open(TRAIN_JSONL, "r") as f:
    for line in f:
        if line.strip():
            rows.append(json.loads(line))

# Convert to arrays
X = np.array([r["x"] for r in rows], dtype=np.float32)
y = np.array([r["y"] for r in rows], dtype=np.int64)

print("Loaded:", len(rows), "examples")
print("X shape:", X.shape, "y shape:", y.shape)
print("Label distribution:", dict(zip(*np.unique(y, return_counts=True))))

In [ ]:
# Train/val/test split

from sklearn.model_selection import train_test_split

# First split off TEST
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=TEST_FRAC, random_state=RANDOM_SEED, stratify=y
)

# Then split remaining into TRAIN / VAL
val_size_of_temp = VAL_FRAC / (1.0 - TEST_FRAC)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=val_size_of_temp, random_state=RANDOM_SEED, stratify=y_temp
)

print("Train:", X_train.shape, "Val:", X_val.shape, "Test:", X_test.shape)

In [ ]:
# Train logistic regression model

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

clf = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_SEED)
)

clf.fit(X_train, y_train)
print("Trained.")

In [ ]:
# Evaluate and save

from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import joblib

def eval_split(name, Xs, ys):
    probs = clf.predict_proba(Xs)[:, 1]
    preds = (probs >= 0.5).astype(int)
    print(f"\n== {name} ==")
    print("AUC:", roc_auc_score(ys, probs))
    print(confusion_matrix(ys, preds))
    print(classification_report(ys, preds, digits=3))
    return probs

_ = eval_split("VAL", X_val, y_val)
_ = eval_split("TEST", X_test, y_test)

# Save model and feature names
joblib.dump(clf, "query_router_logreg.joblib")
with open("query_router_features.json", "w") as f:
    json.dump({"feature_names": feature_names}, f, indent=2)

print("\nSaved: query_router_logreg.joblib, query_router_features.json")